In [45]:
!pip -q install lxml xmltodict 
#xmltodict (xml->dict)

한글 파일 압축 푸는 함수

In [58]:
############## 한글 파일 압축 푸는 함수 #############################
from pathlib import Path
from zipfile import ZipFile


def extract_hwpx(hwpx_path: str | Path, output_dir: str | Path) -> Path:
    """한글 파일 압축 푸는 함수"""
    hwpx_path = Path(hwpx_path)
    output_dir = Path(output_dir)

    if not hwpx_path.exists():
        raise FileNotFoundError(f"hwpx file not found: {hwpx_path}")

    output_dir.mkdir(parents=True, exist_ok=True)

    with ZipFile(hwpx_path, "r") as zip_file:
        zip_file.extractall(output_dir)

    return output_dir

############## 실행 방법 #############################
extract_dir = extract_hwpx(
    r"HWPX 파일\ex_빈파일.hwpx",
    "빈파일_xml"
)

print("압축 해제 완료:", extract_dir)

for file_path in extract_dir.rglob("*"):
    print(file_path)

압축 해제 완료: 빈파일_xml
빈파일_xml\Contents
빈파일_xml\META-INF
빈파일_xml\mimetype
빈파일_xml\Preview
빈파일_xml\settings.xml
빈파일_xml\version.xml
빈파일_xml\Contents\content.hpf
빈파일_xml\Contents\header.xml
빈파일_xml\Contents\section0.xml
빈파일_xml\META-INF\container.rdf
빈파일_xml\META-INF\container.xml
빈파일_xml\META-INF\manifest.xml
빈파일_xml\Preview\PrvImage.png
빈파일_xml\Preview\PrvText.txt


section0.xml, header.xml, settings.xml-> *.json

In [ ]:
# 텍스트·표·수식·레이아웃을 렌더링하려면 실제로 집중해서 분석해야 하는 파일
# section0.xml
# header.xml
# settings.xml

########### section0.xml, header.xml, settings.xml-> *.json ###########
def hwpx_xml_to_json(xml_dir, save_path):
    """ection0.xml, header.xml, settings.xml-> *.json"""
    from pathlib import Path
    import xml.etree.ElementTree as ET
    import json

    xml_dir = Path(xml_dir)

    targets = {
        "section": xml_dir / "Contents" / "section0.xml",
        "header": xml_dir / "Contents" / "header.xml",
        "settings": xml_dir / "settings.xml",
    }

    def strip_ns(tag):
        return tag.split("}", 1)[-1] if "}" in tag else tag

    def xml_to_dict(elem):
        return {
            "tag": strip_ns(elem.tag),
            "attrs": elem.attrib,
            "text": (elem.text or "").strip(),
            "children": [xml_to_dict(child) for child in elem]
        }

    hwpx_jsonb = {}

    for name, path in targets.items():
        tree = ET.parse(path)
        root = tree.getroot()

        hwpx_jsonb[name] = {
            "source_path": str(path),
            "root_tag": strip_ns(root.tag),
            "data": xml_to_dict(root)
        }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(hwpx_jsonb, f, ensure_ascii=False, indent=2)

    print(f"저장 완료: {save_path}")

    return hwpx_jsonb

########### 사용 ###########
hwpx_json = hwpx_xml_to_json(
    xml_dir="빈파일_xml",
    save_path="빈파일_json\빈파일_hwpx.json"
)
print(hwpx_json)

In [71]:
# JSON
# → XML 문자열 생성
# → 기존 HWPX 템플릿 XML 파일 교체
# → 다시 ZIP 압축
# → .hwpx 생성
# json을 xml로 변환 후, 한글템플릿과 치환하여 한글 파일로 변경하는 함수
def json_to_hwpx(json_data, template_dir, output_hwpx):
    """json을 xml로 변환 후, 한글템플릿과 치환하여 한글 파일로 변경하는 함수"""
    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    import xml.etree.ElementTree as ET

    def json_to_xml_elem(data):
        elem = ET.Element(data["tag"], data.get("attrs", {}))

        text = data.get("text", "")
        if text:
            elem.text = text

        for child in data.get("children", []):
            elem.append(json_to_xml_elem(child))

        return elem

    template_dir = Path(template_dir)

    targets = {
        "section": template_dir / "Contents" / "section0.xml",
        "header": template_dir / "Contents" / "header.xml",
        "settings": template_dir / "settings.xml",
    }

    for key, xml_path in targets.items():
        root_data = json_data[key]["data"]
        root_elem = json_to_xml_elem(root_data)

        tree = ET.ElementTree(root_elem)
        ET.indent(tree, space="  ", level=0)
        tree.write(
            xml_path,
            encoding="utf-8",
            xml_declaration=True
        )

        print("XML 치환 완료:", xml_path)

    with ZipFile(output_hwpx, "w", ZIP_DEFLATED) as z:
        for file in template_dir.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(template_dir))

    print("HWPX 생성 완료:", output_hwpx)

    return output_hwpx

json_to_hwpx(
    json_data=hwpx_json,
    template_dir="빈파일_xml",
    output_hwpx="복원_빈파일.hwpx"
)

XML 치환 완료: 빈파일_xml\Contents\section0.xml
XML 치환 완료: 빈파일_xml\Contents\header.xml
XML 치환 완료: 빈파일_xml\settings.xml
HWPX 생성 완료: 복원_빈파일.hwpx


'복원_빈파일.hwpx'

In [ ]:
# 템플릿 HWPX(XML)
# +
# 수정된 XML
# ↓
# 필요한 XML만 치환
# ↓
# 나머지 파일 유지
# ↓
# HWPX 생성
def xml_to_hwpx(
    xml_dir,
    template_dir,
    output_hwpx
):
    """템플릿 HWPX 구조를 유지하고 XML만 치환하여 HWPX 생성"""

    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    from shutil import copytree, copy2, rmtree
    import tempfile

    xml_dir = Path(xml_dir)
    template_dir = Path(template_dir)

    with tempfile.TemporaryDirectory() as temp_dir:

        work_dir = Path(temp_dir) / "hwpx"

        # 1. 템플릿 전체 복사
        copytree(template_dir, work_dir)

        # 2. XML 치환
        replace_files = [
            "Contents/section0.xml",
            "Contents/header.xml",
            "settings.xml",
        ]

        for rel_path in replace_files:

            src = xml_dir / rel_path
            dst = work_dir / rel_path

            if src.exists():
                copy2(src, dst)
                print("XML 치환:", rel_path)

        # 3. HWPX 생성
        with ZipFile(output_hwpx, "w", ZIP_DEFLATED) as z:

            for file in work_dir.rglob("*"):
                if file.is_file():
                    z.write(
                        file,
                        file.relative_to(work_dir)
                    )

    print("HWPX 생성 완료:", output_hwpx)

    return output_hwpx

xml_to_hwpx(
    xml_dir="step2_output/xml",
    template_dir="빈파일_xml",
    output_hwpx="복원.hwpx"
)